# Training de YOLOv8n para detecciÃ³n cenital de cabezas (Phase A)

Phase A del re-entreno del People Counter: fine-tune de `yolov8n` sobre el dataset Roboflow `overhead_person` (o equivalente), y export a ONNX para que despuÃ©s se pueda compilar a un `.hef` de Hailo-8L en un workstation (WSL2 + Hailo Dataflow Compiler).

**Runtime:** Colab â†’ `Runtime` â†’ `Change runtime type` â†’ `T4 GPU` (free tier).

**Colab Secrets requeridos** (Ã­cono ðŸ”‘ en el sidebar):
- `ROBOFLOW_API_KEY`

**Ediciones manuales antes de correr:**
1. Editar `WORKSPACE`, `PROJECT`, `VERSION` en la cell 4 (matchea la URL de tu dataset de Roboflow Universe).
2. (opcional) Ajustar `EPOCHS`, `BATCH`, `IMGSZ` en la cell 5.

Wall-time total en T4: ~2-4 h para ~5k imÃ¡genes Ã— 50 epochs. Los checkpoints se guardan en Drive cada 10 epochs asÃ­ que una desconexiÃ³n es recuperable.

## 1. Montar Google Drive (para persistir checkpoints)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_ROOT = '/content/drive/MyDrive/people-counter-training'
os.makedirs(DRIVE_ROOT, exist_ok=True)
print('Workspace en Drive:', DRIVE_ROOT)

## 2. Instalar dependencias

In [ ]:
!pip install -q ultralytics==8.3.* roboflow==1.1.*
import ultralytics; ultralytics.checks()

## 3. Leer la API key de Roboflow desde Colab Secrets

AgregÃ¡ `ROBOFLOW_API_KEY` a Colab Secrets (Ã­cono ðŸ”‘ en el sidebar izquierdo) para que no quede hardcodeada en el notebook.

In [ ]:
from google.colab import userdata
ROBOFLOW_API_KEY = userdata.get('ROBOFLOW_API_KEY')
assert ROBOFLOW_API_KEY, 'AgregÃ¡ ROBOFLOW_API_KEY a Colab Secrets y corrÃ© esta cell de nuevo.'
print('API key cargada (length =', len(ROBOFLOW_API_KEY), ')')

## 4. Bajar el dataset de Roboflow

EditÃ¡ los tres slugs de abajo para que matcheen la URL de tu dataset:
`https://universe.roboflow.com/<WORKSPACE>/<PROJECT>/<VERSION>`

In [ ]:
WORKSPACE = 'REPLACE-ME-workspace-slug'
PROJECT   = 'REPLACE-ME-project-slug'
VERSION   = 1

from roboflow import Roboflow
rf = Roboflow(api_key=ROBOFLOW_API_KEY)
project = rf.workspace(WORKSPACE).project(PROJECT)
dataset = project.version(VERSION).download('yolov8',
    location=f'/content/dataset/{WORKSPACE}__{PROJECT}__v{VERSION}',
    overwrite=True)
DATA_YAML = f'{dataset.location}/data.yaml'
print('Dataset en:', dataset.location)
print('data.yaml :', DATA_YAML)
!cat "$DATA_YAML"

## 5. Entrenar

Arrancamos desde `yolov8n.pt` (pretrained en COCO â€” tiene clase `person` que es un prior Ãºtil aunque vayamos a re-targetearlo a head/overhead). Los pesos head-class de Owen718 (entrenados en CrowdHuman) serÃ­an un starting point mÃ¡s fuerte pero requieren bajar a mano el release de su repo de GitHub; si querÃ©s cambiar, reemplazÃ¡ el argumento `model=` con el path a esos pesos.

In [ ]:
EPOCHS = 50
BATCH  = 16        # T4 (16GB) OOMea con 32 sobre datasets ~15k+ imágenes; si OOMea bajá a 8
IMGSZ  = 640
PROJECT_DIR = f'{DRIVE_ROOT}/runs'
RUN_NAME    = f'{PROJECT}_v{VERSION}_yolov8n'

from ultralytics import YOLO
model = YOLO('yolov8n.pt')   # arranque desde COCO pretrained

results = model.train(
    data=DATA_YAML,
    epochs=EPOCHS,
    batch=BATCH,
    imgsz=IMGSZ,
    project=PROJECT_DIR,
    name=RUN_NAME,
    exist_ok=True,
    save_period=10,        # checkpoint cada 10 epochs (resiliente a desconexiÃ³n de Drive)
    patience=20,           # early-stop si la val mAP deja de mejorar
    pretrained=True,
    optimizer='AdamW',
    lr0=0.001,
    seed=42,
    verbose=True,
)
BEST_PT = f'{PROJECT_DIR}/{RUN_NAME}/weights/best.pt'
print('Best weights en:', BEST_PT)

## 6. Validar

In [ ]:
from ultralytics import YOLO
best = YOLO(BEST_PT)
metrics = best.val(data=DATA_YAML, imgsz=IMGSZ, plots=True)
print('mAP50    :', float(metrics.box.map50))
print('mAP50-95 :', float(metrics.box.map))
print('Precision:', float(metrics.box.mp))
print('Recall   :', float(metrics.box.mr))

## 7. Exportar a ONNX (para compilaciÃ³n a Hailo)

Hailo Dataflow Compiler (`hailomz compile`) toma ONNX de input. Exportamos con `opset=12` (compatible con Hailo) y dynamic batch off (HEF necesita un grafo de tamaÃ±o fijo).

In [ ]:
best.export(format='onnx', imgsz=IMGSZ, opset=12, dynamic=False, simplify=True)
ONNX_PATH = BEST_PT.replace('.pt', '.onnx')
print('ONNX:', ONNX_PATH)
import os; print('size (MB):', round(os.path.getsize(ONNX_PATH) / (1024*1024), 2))

## 8. Stash del ONNX + calibration set en Drive

Copiamos tambiÃ©n ~200 imÃ¡genes random del training set a Drive â€” el compilador de Hailo las usa como calibration set para la cuantizaciÃ³n a int8 cuando compila el HEF.

In [ ]:
import shutil, random
EXPORT_DIR = f'{DRIVE_ROOT}/export/{RUN_NAME}'
os.makedirs(EXPORT_DIR, exist_ok=True)

shutil.copy(ONNX_PATH, f'{EXPORT_DIR}/best.onnx')
shutil.copy(BEST_PT,   f'{EXPORT_DIR}/best.pt')
shutil.copy(DATA_YAML, f'{EXPORT_DIR}/data.yaml')

# Calibration set
calib_dir = f'{EXPORT_DIR}/calib'
os.makedirs(calib_dir, exist_ok=True)
import glob
train_imgs = glob.glob(f'{dataset.location}/train/images/*')
random.seed(42)
for src in random.sample(train_imgs, min(200, len(train_imgs))):
    shutil.copy(src, calib_dir)

print('Stash en Drive:', EXPORT_DIR)
!ls -la "$EXPORT_DIR"

## PrÃ³ximo paso (off-Colab): compilar a HEF de Hailo

BajÃ¡ `EXPORT_DIR` desde Drive a tu workstation con WSL2 Ubuntu, y despuÃ©s:

```bash
# Adentro de WSL2 con hailo-dataflow-compiler instalado
hailomz compile yolov8n \
    --ckpt best.onnx \
    --hw-arch hailo8l \
    --calib-path calib/
```

Output: `yolov8n.hef`. SCP a la Pi y actualizÃ¡ `detection.model_path` en `/etc/people-counter/config.yaml`.